# TECHNICAL DATA INSPECTION NOTEBOOK
**HealthConnect Experience Lab — Data Analytics Track | Week 4**  
**Nancy Lee YIMBERE ALAPINI | Performance & Decision Intelligence Analyst**

> **Role of this notebook:** Supporting Technical Evidence for the Week 4 Initial Analysis Document.

This notebook provides reproducible technical evidence for the Week 4 dataset overview and initial data quality assessment. It is intentionally limited to **data understanding, inspection, validation, and analytical readiness**. It does **not** answer the Business Questions, test the full analytical hypotheses, calculate the proposed KPIs for decision-making, or build a final dashboard.

**Documentation standard:** `Purpose → Code → Result → Interpretation`


## 1. Notebook Context & Objective

### Purpose
The objective is to:
- load the original HealthConnect appointment dataset and Data Dictionary;
- verify the documented schema against the observed dataset;
- validate the appointment-level grain;
- assess data types, missing values, duplicates, value validity, and cross-field consistency;
- assess consistency across repeated `patient_id` values;
- consolidate identified issues in a Data Quality Issue Register;
- determine analytical readiness for the next project stage.

The original source files are not modified. Any future cleaned or transformed dataset should be saved separately.


## 2. Environment Setup & Data Loading

### Purpose
Load the libraries and source files required for reproducible inspection. The notebook first searches the working directory, then `/mnt/data`, so it remains portable when the source files are stored alongside the notebook.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

DATASET_NAME = "HealthConnect_Appointment_Data.csv"
DICTIONARY_NAME = "HealthConnect_Data_Dictionary.xlsx"

def locate_file(filename):
    candidates = [
        Path.cwd() / filename,
        Path("/mnt/data") / filename,
        Path("/mnt/data/healthconnect") / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    matches = list(Path("/mnt/data").rglob(filename))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"{filename} was not found.")

dataset_path = locate_file(DATASET_NAME)
dictionary_path = locate_file(DICTIONARY_NAME)

appointments = pd.read_csv(dataset_path)
data_dictionary = pd.read_excel(dictionary_path)

print(f"Dataset loaded: {DATASET_NAME}")
print(f"Data Dictionary loaded: {DICTIONARY_NAME}")
print(f"Dataset shape: {appointments.shape}")


Dataset loaded: HealthConnect_Appointment_Data.csv
Data Dictionary loaded: HealthConnect_Data_Dictionary.xlsx
Dataset shape: (5000, 18)


### Interpretation
Successful loading confirms that the original source files are available for reproducible inspection. No transformation has been applied at this stage.


## 3. Data Dictionary Alignment

### Purpose
Verify that the variables documented in the Data Dictionary correspond to the columns actually present in the appointment dataset.


In [2]:
expected_columns = data_dictionary["Variable"].astype(str).tolist()
observed_columns = appointments.columns.tolist()

missing_from_dataset = sorted(set(expected_columns) - set(observed_columns))
unexpected_in_dataset = sorted(set(observed_columns) - set(expected_columns))

alignment_summary = pd.DataFrame({
    "Metric": [
        "Variables documented in Data Dictionary",
        "Columns observed in dataset",
        "Documented variables missing from dataset",
        "Unexpected dataset columns"
    ],
    "Result": [
        len(expected_columns),
        len(observed_columns),
        len(missing_from_dataset),
        len(unexpected_in_dataset)
    ]
})
display(alignment_summary)

print("Missing documented variables:", missing_from_dataset)
print("Unexpected columns:", unexpected_in_dataset)


,Metric,Result
0,Variables documented in Data Dictionary,18
1,Columns observed in dataset,18
2,Documented variables missing from dataset,0
3,Unexpected dataset columns,0


Missing documented variables: []
Unexpected columns: []


### Interpretation
The dataset contains the expected variables documented in the Data Dictionary when both counts match and no missing or unexpected fields are returned. This check establishes schema completeness before deeper quality assessment.


## 4. Dataset Structure & Grain Validation

### Purpose
Confirm dataset dimensions, date coverage, record identifiers, and whether the expected grain is **one row per appointment**.


In [3]:
# Parse copies of date fields for validation without overwriting raw columns.
booking_date_parsed = pd.to_datetime(appointments["booking_date"], errors="coerce")
appointment_date_parsed = pd.to_datetime(appointments["appointment_date"], errors="coerce")

structure_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Unique appointment_id",
        "Unique patient_id",
        "Appointment date start",
        "Appointment date end",
        "Booking date start",
        "Booking date end"
    ],
    "Result": [
        len(appointments),
        appointments.shape[1],
        appointments["appointment_id"].nunique(dropna=True),
        appointments["patient_id"].nunique(dropna=True),
        appointment_date_parsed.min().date(),
        appointment_date_parsed.max().date(),
        booking_date_parsed.min().date(),
        booking_date_parsed.max().date(),
    ]
})
display(structure_summary)

print("First 5 records:")
display(appointments.head())


,Metric,Result
0,Rows,5000
1,Columns,18
2,Unique appointment_id,5000
3,Unique patient_id,1696
4,Appointment date start,2025-01-01
5,Appointment date end,2026-06-30
6,Booking date start,2024-11-07
7,Booking date end,2026-06-27


First 5 records:


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


### Interpretation
If `appointment_id` is unique for every row, the expected appointment-level grain is supported. Repeated `patient_id` values are permissible because the Data Dictionary indicates that one anonymised patient identifier may appear across multiple appointments. Appointment-level and patient-level populations must therefore remain distinct in future analysis.


## 5. Data Type Assessment

### Purpose
Compare observed pandas data types with the expected Data Dictionary types and identify preparation requirements without treating routine CSV parsing behaviour as a data-quality defect.


In [4]:
observed_types = appointments.dtypes.astype(str).rename("Observed pandas dtype").reset_index()
observed_types.columns = ["Variable", "Observed pandas dtype"]

type_review = data_dictionary[["Variable", "Data Type"]].merge(observed_types, on="Variable", how="left")
display(type_review)

date_parse_check = pd.DataFrame({
    "Variable": ["booking_date", "appointment_date"],
    "Non-null raw values": [
        appointments["booking_date"].notna().sum(),
        appointments["appointment_date"].notna().sum()
    ],
    "Successfully parsed as date": [
        booking_date_parsed.notna().sum(),
        appointment_date_parsed.notna().sum()
    ]
})
display(date_parse_check)


,Variable,Data Type,Observed pandas dtype
0,appointment_id,Text,object
1,patient_id,Text,object
2,gender,Text,object
3,age,Integer,int64
4,age_group,Text,object
5,appointment_type,Text,object
6,booking_date,Date,object
7,appointment_date,Date,object
8,appointment_day,Text,object
9,appointment_time,Text,object


,Variable,Non-null raw values,Successfully parsed as date
0,booking_date,5000,5000
1,appointment_date,5000,5000


### Interpretation
CSV date fields are commonly imported as text/object values. If all non-null date values parse successfully, this is treated as a **data preparation requirement** rather than a data-quality defect. Date fields should be explicitly converted before subsequent temporal analysis.


## 6. Missing Values Assessment

### Purpose
Quantify missingness by variable and distinguish genuine incompleteness from structurally Not Applicable values.


In [5]:
missing_summary = (
    appointments.isna()
    .sum()
    .to_frame("Missing_Count")
    .assign(Missing_Pct=lambda x: (x["Missing_Count"] / len(appointments) * 100).round(2))
    .query("Missing_Count > 0")
    .sort_values("Missing_Count", ascending=False)
)
display(missing_summary)

# Reminder structural-missingness check
reminder_check = pd.DataFrame({
    "Check": [
        "reminder_sent = No AND reminder_channel is missing",
        "reminder_sent = No AND reminder_channel is present",
        "reminder_sent = Yes AND reminder_channel is missing",
        "reminder_sent = Yes AND reminder_channel is present"
    ],
    "Count": [
        ((appointments["reminder_sent"] == "No") & appointments["reminder_channel"].isna()).sum(),
        ((appointments["reminder_sent"] == "No") & appointments["reminder_channel"].notna()).sum(),
        ((appointments["reminder_sent"] == "Yes") & appointments["reminder_channel"].isna()).sum(),
        ((appointments["reminder_sent"] == "Yes") & appointments["reminder_channel"].notna()).sum(),
    ]
})
display(reminder_check)


,Missing_Count,Missing_Pct
reminder_channel,1366,27.32
distance_to_clinic_km,90,1.80
waiting_time_minutes,60,1.20


,Check,Count
0,reminder_sent = No AND reminder_channel is mis...,1366
1,reminder_sent = No AND reminder_channel is pre...,0
2,reminder_sent = Yes AND reminder_channel is mi...,0
3,reminder_sent = Yes AND reminder_channel is pr...,3634


### Interpretation
Missing `reminder_channel` values are treated as **structurally Not Applicable** when they occur only where `reminder_sent = No`. Missing values in `distance_to_clinic_km` and `waiting_time_minutes` represent limited incompleteness that should be assessed before those variables are used analytically. No automatic imputation decision is made in Week 4.


## 7. Duplicate & Identifier Checks

### Purpose
Assess exact duplicate rows, primary-key uniqueness, and the presence of repeated patient identifiers.


In [6]:
duplicate_summary = pd.DataFrame({
    "Check": [
        "Exact duplicate rows",
        "Duplicated appointment_id values",
        "Missing appointment_id values",
        "Unique patient_id values",
        "patient_id values appearing in more than one record"
    ],
    "Result": [
        appointments.duplicated().sum(),
        appointments["appointment_id"].duplicated().sum(),
        appointments["appointment_id"].isna().sum(),
        appointments["patient_id"].nunique(dropna=True),
        (appointments.groupby("patient_id").size() > 1).sum()
    ]
})
display(duplicate_summary)


,Check,Result
0,Exact duplicate rows,0
1,Duplicated appointment_id values,0
2,Missing appointment_id values,0
3,Unique patient_id values,1696
4,patient_id values appearing in more than one r...,1394


### Interpretation
A complete and unique `appointment_id` supports use of the field as the appointment-level record identifier. Repeated `patient_id` values are not duplicates by themselves and should not be removed solely because the same patient identifier appears across multiple appointments.


## 8. Categorical & Numerical Validity Checks

### Purpose
Inspect categorical modalities and numerical ranges for obvious invalid or unexpected values without interpreting their relationship with no-show outcomes.


In [7]:
categorical_columns = [
    "gender", "age_group", "appointment_type", "appointment_day",
    "appointment_time", "reminder_sent", "reminder_channel", "appointment_outcome"
]

for col in categorical_columns:
    print(f"\n{col}:")
    print(appointments[col].value_counts(dropna=False).sort_index())

numeric_columns = [
    "age", "booking_lead_days", "previous_appointments",
    "previous_no_shows", "distance_to_clinic_km", "waiting_time_minutes"
]

numeric_range = appointments[numeric_columns].agg(["min", "max"]).T
numeric_range.columns = ["Minimum", "Maximum"]
display(numeric_range)

negative_counts = pd.DataFrame({
    "Variable": numeric_columns,
    "Negative_Count": [(appointments[c].dropna() < 0).sum() for c in numeric_columns]
})
display(negative_counts)



gender:
gender
Female               2488
Male                 2404
Prefer not to say     108
Name: count, dtype: int64

age_group:
age_group
18-24     564
25-34     783
35-44     819
45-54     793
55-64     800
65+      1241
Name: count, dtype: int64

appointment_type:
appointment_type
Diagnostic Test             593
Follow-up                  1421
General Consultation       2086
Specialist Consultation     900
Name: count, dtype: int64

appointment_day:
appointment_day
Friday       728
Monday       701
Saturday     724
Sunday       737
Thursday     684
Tuesday      689
Wednesday    737
Name: count, dtype: int64

appointment_time:
appointment_time
Afternoon    2094
Evening       679
Morning      2227
Name: count, dtype: int64

reminder_sent:
reminder_sent
No     1366
Yes    3634
Name: count, dtype: int64

reminder_channel:
reminder_channel
Email        533
SMS         2000
WhatsApp    1101
NaN         1366
Name: count, dtype: int64

appointment_outcome:
appointment_outcome
Attended   

,Minimum,Maximum
age,18.0,80.0
booking_lead_days,0.0,60.0
previous_appointments,0.0,11.0
previous_no_shows,0.0,5.0
distance_to_clinic_km,0.5,45.0
waiting_time_minutes,2.0,68.0


,Variable,Negative_Count
0,age,0
1,booking_lead_days,0
2,previous_appointments,0
3,previous_no_shows,0
4,distance_to_clinic_km,0
5,waiting_time_minutes,0


### Interpretation
This section validates the observed domain of each field only. It does not determine whether a particular value is operationally desirable or undesirable. For example, the maximum recorded waiting time is a descriptive validity observation, not a clinical or service-quality benchmark.


## 9. Cross-Field Consistency Checks

### Purpose
Test explicit or derivable consistency rules identified during the Data Dictionary Review.


In [8]:
def expected_age_group(age):
    if pd.isna(age):
        return np.nan
    if 18 <= age <= 24:
        return "18-24"
    if 25 <= age <= 34:
        return "25-34"
    if 35 <= age <= 44:
        return "35-44"
    if 45 <= age <= 54:
        return "45-54"
    if 55 <= age <= 64:
        return "55-64"
    if age >= 65:
        return "65+"
    return np.nan

derived_age_group = appointments["age"].apply(expected_age_group)
derived_appointment_day = appointment_date_parsed.dt.day_name()
derived_lead_days = (appointment_date_parsed - booking_date_parsed).dt.days

cross_field_checks = pd.DataFrame({
    "Validation Rule": [
        "age is consistent with age_group",
        "appointment_date is consistent with appointment_day",
        "appointment_date - booking_date equals booking_lead_days",
        "previous_no_shows <= previous_appointments",
        "reminder_sent = No implies reminder_channel is missing",
        "reminder_sent = Yes implies reminder_channel is present",
        "booking_date <= appointment_date"
    ],
    "Violation_Count": [
        (derived_age_group != appointments["age_group"]).sum(),
        (derived_appointment_day != appointments["appointment_day"]).sum(),
        (derived_lead_days != appointments["booking_lead_days"]).sum(),
        (appointments["previous_no_shows"] > appointments["previous_appointments"]).sum(),
        ((appointments["reminder_sent"] == "No") & appointments["reminder_channel"].notna()).sum(),
        ((appointments["reminder_sent"] == "Yes") & appointments["reminder_channel"].isna()).sum(),
        (booking_date_parsed > appointment_date_parsed).sum()
    ]
})
display(cross_field_checks)


,Validation Rule,Violation_Count
0,age is consistent with age_group,0
1,appointment_date is consistent with appointmen...,0
2,appointment_date - booking_date equals booking...,0
3,previous_no_shows <= previous_appointments,0
4,reminder_sent = No implies reminder_channel is...,0
5,reminder_sent = Yes implies reminder_channel i...,0
6,booking_date <= appointment_date,0


### Interpretation
Zero violations support strong **row-level consistency** for the tested rules. Row-level consistency should not, however, be confused with longitudinal consistency across repeated patient identifiers, which is assessed separately below.


## 10. Repeated `patient_id` Consistency Checks

### Purpose
Assess whether repeated `patient_id` values behave like stable longitudinal patient identities. This is essential before using demographic or historical fields for patient-level longitudinal interpretation.


In [9]:
patient_gender_nunique = appointments.groupby("patient_id")["gender"].nunique(dropna=True)
patient_age_nunique = appointments.groupby("patient_id")["age"].nunique(dropna=True)
patient_agegroup_nunique = appointments.groupby("patient_id")["age_group"].nunique(dropna=True)
patient_age_range = appointments.groupby("patient_id")["age"].agg(lambda s: s.max() - s.min())

patient_consistency = pd.DataFrame({
    "Check": [
        "Patient IDs with multiple recorded gender categories",
        "Patient IDs with multiple recorded ages",
        "Patient IDs with multiple recorded age groups",
        "Patient IDs with recorded age range > 2 years"
    ],
    "Count": [
        (patient_gender_nunique > 1).sum(),
        (patient_age_nunique > 1).sum(),
        (patient_agegroup_nunique > 1).sum(),
        (patient_age_range > 2).sum()
    ]
})
display(patient_consistency)

print("Maximum observed age range within a repeated patient_id:", patient_age_range.max())


,Check,Count
0,Patient IDs with multiple recorded gender cate...,1058
1,Patient IDs with multiple recorded ages,1385
2,Patient IDs with multiple recorded age groups,1287
3,Patient IDs with recorded age range > 2 years,1343


Maximum observed age range within a repeated patient_id: 62


### Historical-count chronology check

To avoid arbitrary ordering among multiple appointments on the same date, the chronology check first aggregates each patient's historical counters to the **maximum recorded value per appointment date**, then checks whether that date-level series decreases on a later date.


In [10]:
chronology = appointments.copy()
chronology["appointment_date_parsed"] = pd.to_datetime(chronology["appointment_date"], errors="coerce")

date_level_history = (
    chronology.groupby(["patient_id", "appointment_date_parsed"], as_index=False)
    [["previous_appointments", "previous_no_shows"]]
    .max()
    .sort_values(["patient_id", "appointment_date_parsed"])
)

def has_later_decrease(series):
    prior_max = series.cummax().shift(1)
    return (series < prior_max).fillna(False).any()

previous_appointments_decrease = (
    date_level_history.groupby("patient_id")["previous_appointments"]
    .apply(has_later_decrease)
)
previous_no_shows_decrease = (
    date_level_history.groupby("patient_id")["previous_no_shows"]
    .apply(has_later_decrease)
)

history_consistency = pd.DataFrame({
    "Check": [
        "Patient IDs with a later decrease in recorded previous_appointments",
        "Patient IDs with a later decrease in recorded previous_no_shows"
    ],
    "Count": [
        int(previous_appointments_decrease.sum()),
        int(previous_no_shows_decrease.sum())
    ]
})
display(history_consistency)


,Check,Count
0,Patient IDs with a later decrease in recorded ...,991
1,Patient IDs with a later decrease in recorded ...,757


### Interpretation
The repeated-patient checks show that `patient_id` should **not automatically be treated as a reliable longitudinal identity key**. The dataset remains usable at the appointment-record level, but demographic and historical fields linked to repeated patient identifiers require cautious interpretation. The notebook does not attempt to repair these inconsistencies by imposing a first, last, or most-frequent patient attribute because the source does not identify which value should be considered correct.


## 11. Data Quality Issue Register

### Purpose
Consolidate the most decision-relevant data-quality findings, their evidence, and their planned analytical handling.


In [11]:
dq_register = pd.DataFrame([
    {
        "Issue": "Dataset schema alignment",
        "Evidence": f"{len(expected_columns)}/{len(expected_columns)} documented variables present",
        "Risk": "None",
        "Proposed_Treatment": "Retain",
        "Analytical_Impact": "Supports documented dataset structure"
    },
    {
        "Issue": "appointment_id uniqueness",
        "Evidence": f"{appointments['appointment_id'].nunique()} unique IDs across {len(appointments)} rows",
        "Risk": "None",
        "Proposed_Treatment": "Retain",
        "Analytical_Impact": "Supports appointment-level grain"
    },
    {
        "Issue": "Date fields imported as text",
        "Evidence": "All non-null booking_date and appointment_date values are parseable",
        "Risk": "Low / technical",
        "Proposed_Treatment": "Convert during data preparation",
        "Analytical_Impact": "Required before temporal analysis"
    },
    {
        "Issue": "Structural reminder_channel missingness",
        "Evidence": f"{appointments['reminder_channel'].isna().sum()} missing values; all correspond to reminder_sent = No",
        "Risk": "None / Not Applicable",
        "Proposed_Treatment": "Treat as Not Applicable",
        "Analytical_Impact": "Reminder-channel analysis restricted to reminder_sent = Yes"
    },
    {
        "Issue": "Missing distance_to_clinic_km",
        "Evidence": f"{appointments['distance_to_clinic_km'].isna().sum()} records ({appointments['distance_to_clinic_km'].isna().mean()*100:.2f}%)",
        "Risk": "Moderate",
        "Proposed_Treatment": "Assess missingness before analytical use; no automatic imputation",
        "Analytical_Impact": "May reduce usable population for accessibility analysis"
    },
    {
        "Issue": "Missing waiting_time_minutes",
        "Evidence": f"{appointments['waiting_time_minutes'].isna().sum()} records ({appointments['waiting_time_minutes'].isna().mean()*100:.2f}%)",
        "Risk": "Moderate",
        "Proposed_Treatment": "Assess missingness before analytical use; no automatic imputation",
        "Analytical_Impact": "May reduce usable population for operational-context analysis"
    },
    {
        "Issue": "Patient demographic consistency",
        "Evidence": f"{(patient_gender_nunique > 1).sum()} patient IDs have multiple genders; {(patient_age_nunique > 1).sum()} have multiple ages",
        "Risk": "High for longitudinal interpretation",
        "Proposed_Treatment": "Flag limitation; do not impute arbitrary patient profiles",
        "Analytical_Impact": "Use recorded demographic attributes at appointment-record level"
    },
    {
        "Issue": "Recorded historical consistency",
        "Evidence": f"{int(previous_appointments_decrease.sum())} patient IDs show later decreases in previous_appointments; {int(previous_no_shows_decrease.sum())} in previous_no_shows",
        "Risk": "High for longitudinal interpretation",
        "Proposed_Treatment": "Flag limitation; do not reconstruct longitudinal history",
        "Analytical_Impact": "Use historical fields as recorded appointment-level information, with caution"
    }
])

display(dq_register)


,Issue,Evidence,Risk,Proposed_Treatment,Analytical_Impact
0,Dataset schema alignment,18/18 documented variables present,None,Retain,Supports documented dataset structure
1,appointment_id uniqueness,5000 unique IDs across 5000 rows,None,Retain,Supports appointment-level grain
2,Date fields imported as text,All non-null booking_date and appointment_date...,Low / technical,Convert during data preparation,Required before temporal analysis
3,Structural reminder_channel missingness,1366 missing values; all correspond to reminde...,None / Not Applicable,Treat as Not Applicable,Reminder-channel analysis restricted to remind...
4,Missing distance_to_clinic_km,90 records (1.80%),Moderate,Assess missingness before analytical use; no a...,May reduce usable population for accessibility...
5,Missing waiting_time_minutes,60 records (1.20%),Moderate,Assess missingness before analytical use; no a...,May reduce usable population for operational-c...
6,Patient demographic consistency,1058 patient IDs have multiple genders; 1385 h...,High for longitudinal interpretation,Flag limitation; do not impute arbitrary patie...,Use recorded demographic attributes at appoint...
7,Recorded historical consistency,991 patient IDs show later decreases in previo...,High for longitudinal interpretation,Flag limitation; do not reconstruct longitudin...,Use historical fields as recorded appointment-...


## 12. Analytical Readiness Assessment

### Purpose
Translate technical inspection results into clear guidance for subsequent analytical use without beginning the Business Question analysis.


In [12]:
readiness = pd.DataFrame([
    ["appointment_outcome", "Ready", "Core outcome; expected categories observed"],
    ["appointment_type / appointment_day / appointment_time / booking_lead_days", "Ready", "Row-level checks passed"],
    ["reminder_sent", "Ready", "Complete and internally consistent"],
    ["reminder_channel", "Ready with condition", "Analyse only among reminder_sent = Yes"],
    ["distance_to_clinic_km", "Ready with caution", "Limited missingness requires explicit handling"],
    ["waiting_time_minutes", "Ready with caution", "Limited missingness requires explicit handling"],
    ["age / age_group / gender", "Ready with caution", "Use as recorded appointment-level attributes; longitudinal inconsistency identified"],
    ["previous_appointments / previous_no_shows", "Ready with caution", "Use recorded values only; longitudinal reconstruction not reliable"],
    ["patient_id", "Supporting only", "Useful for structure/counting; not assumed to be a reliable longitudinal identity key"],
    ["appointment_id", "Ready — identifier", "Supports appointment-level grain; not an explanatory variable"]
], columns=["Variable_or_Group", "Readiness_Status", "Rationale"])

display(readiness)


,Variable_or_Group,Readiness_Status,Rationale
0,appointment_outcome,Ready,Core outcome; expected categories observed
1,appointment_type / appointment_day / appointme...,Ready,Row-level checks passed
2,reminder_sent,Ready,Complete and internally consistent
3,reminder_channel,Ready with condition,Analyse only among reminder_sent = Yes
4,distance_to_clinic_km,Ready with caution,Limited missingness requires explicit handling
5,waiting_time_minutes,Ready with caution,Limited missingness requires explicit handling
6,age / age_group / gender,Ready with caution,Use as recorded appointment-level attributes; ...
7,previous_appointments / previous_no_shows,Ready with caution,Use recorded values only; longitudinal reconst...
8,patient_id,Supporting only,Useful for structure/counting; not assumed to ...
9,appointment_id,Ready — identifier,Supports appointment-level grain; not an expla...


### Interpretation
The dataset is sufficiently usable for the planned **appointment-level** analytical work, subject to explicit handling of missing operational variables and clear limitations on patient-level longitudinal interpretation. These constraints should remain visible in future Business Question analysis, KPI definitions, and reporting.


## 13. Notebook Conclusion

The Week 4 technical inspection confirms that the HealthConnect dataset is structurally complete and supports an appointment-level analytical grain. The documented row-level consistency checks are strong, and no exact duplicate rows or duplicated appointment identifiers are identified.

The main analytical cautions are:
1. limited missingness in `distance_to_clinic_km` and `waiting_time_minutes`;
2. structural `reminder_channel` missingness that should be treated as Not Applicable when no reminder was sent;
3. patient-level inconsistencies across repeated `patient_id` values, including demographic and recorded historical attributes.

These findings do not prevent the planned appointment-level analysis, but they define important interpretation boundaries. The next project stage should therefore proceed with explicit population definitions, consistent metric definitions, documented missing-value handling, and caution around any longitudinal patient-level claim.

> **Week 4 boundary maintained:** this notebook establishes technical readiness and evidence. It does not yet answer the Business Questions or test the proposed analytical hypotheses.
